# AH-1S JSBSim RL — Colab defteri

Final sistemi Colab'da çalıştırmak için gereken adımlar:

1. **Kurulum** — fork'u klonla / güncelle, paketleri kur
2. **Hızlı kontrol** — modeller sağlam mı, kod derleniyor mu, kontrol yığını yükleniyor mu
3. **Canlı dashboard** — Stage 1 → Stage 2 → istediğin heading'e dönüş (aynı JSBSim FDM)
4. **Doğrulama testleri** *(isteğe bağlı)*
5. **GIF görselleştirme** *(isteğe bağlı)*

Mevcut modelleri yeniden eğitmek için `training/` klasörüne bak (README'de sırası var).

## 1. Kurulum

`REPO_URL` satırına kendi fork'unun adresini yaz. Temizlik branch'ini `main`'e almadan önce denemek için `BRANCH = "cleanup"` yap.

In [ ]:
REPO_URL = "https://github.com/<KULLANICI_ADIN>/ah1s-rl-project.git"  # <- kendi fork'un
BRANCH = "main"
REPO_DIR = "/content/ah1s-rl-project"

import os
import subprocess

if "<KULLANICI_ADIN>" in REPO_URL:
    raise ValueError("REPO_URL içindeki <KULLANICI_ADIN> kısmına kendi GitHub kullanıcı adını yaz.")

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH], check=True)

%cd {REPO_DIR}
!pip -q install -r requirements.txt

## 2. Hızlı kontrol (birkaç saniye)

Model dosyalarının bozulmadığını (SHA-256), tüm Python dosyalarının derlendiğini ve Stage 1/Stage 2 + turn stack'in yüklendiğini kontrol eder.

In [ ]:
!sha256sum -c models_sha256.txt
!python -m compileall -q . && echo "OK: tüm .py dosyaları derlendi"
!python -c "import validate_final_continuous_mission_v1 as m; m.arb.load_stack(); print('OK: Stage 1/2 modelleri ve turn stack yüklendi')"

## 3. Canlı hedef-heading dashboard

Stage 1 (kalkış → 300 ft hover) ve Stage 2 (ileri uçuş) aynı JSBSim FDM üzerinde başlar. Panelde **FORWARD / READY** görününce `Target Heading` alanına 0–359° arası bir değer gir ve **Fly to Heading**'e bas. Sistem, o anki heading'den hedefe en kısa relatif dönüşü hesaplar (ör. 350° → 10° = +20°). Durdurmak için **Stop**.

In [ ]:
%run run_colab_live_heading_dashboard.py

## 4. Doğrulama testleri (isteğe bağlı)

**Sürekli görev** — Stage 1 → Stage 2 → sırayla verilen relative turn komutları, hepsi aynı FDM'de. Sonda `FINAL CONTINUOUS MISSION: PASS` beklenir.

In [ ]:
!python validate_final_continuous_mission_v1.py 20 -30 75 -90

**Turn robustness** — −50°, +50°, +200°, +360° hedefleri × 5 farklı giriş durumu = 20 koşu. Beklenen: `RANDOMIZED-ENTRY TOTAL: PASS=20/20`. Uzun sürer.

In [ ]:
!python test_turn_full_entry_v22_v21_runtime.py

## 5. GIF görselleştirme (isteğe bağlı)

Aynı final kontrol yığınıyla görevi uçurur ve `ah1s_final_multiturn.gif` üretir.

In [ ]:
!python visualize_final_multiturn.py 20 -30 75 --output ah1s_final_multiturn.gif

from IPython.display import Image, display
display(Image(filename="ah1s_final_multiturn.gif"))